# From interference and CHSH to verified quantum teleportation

Connect three operational tests: single-qubit interference, an ideal CHSH violation, and teleportation of six distinct states. Implement actual mid-circuit measurements and conditional corrections in a local simulator. The expected evidence is a fringe curve, the value 2√2 for the ideal CHSH expression, and unit fidelity for all six corrected output states.

**Prerequisites:** Python 3.12 or later and the tested environment in the [collection README](../../../README.md). Install the [root requirements](../../../requirements.txt) into an isolated environment before opening this notebook; select that environment's kernel. Run all cells from a fresh kernel in order. This notebook makes no network calls, reads no external files, and writes no artifacts by default.

**Reproducibility:** all examples and data here are synthetic. Simulator and sampling seeds are explicit. Assertions distinguish mathematical identities, finite-shot checks, and integration contracts. Approximate comparisons state their tolerance.


Copyright IBM Corp. 2026

SPDX-License-Identifier: Apache-2.0

**License and AI-assistance notice.** Source code is licensed under Apache License 2.0. Modified for open-source preparation on 2026-10-23. This notebook may include contributions involving Anthropic Claude Opus 4+, Claude Sonnet 3.7+, and OpenAI GPT-4+ models; exact per-file model provenance is unverified. See [AI disclosure](../../../AI_DISCLOSURE.md), [license](../../../LICENSE), and [attribution](../../../NOTICE.md). Human contributors remain responsible for review, rights and DCO certification.


## 1. Interference is coherence, not a classical mixture

For the circuit $H R_z(\phi) H|0\rangle$, the final probability of zero is $(1+\cos\phi)/2$. Dephasing in the Z basis after the first H destroys the off-diagonal density-matrix elements and removes the fringe. This models ignoring a which-path measurement result, not a literal optical apparatus.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import DensityMatrix, Operator, Statevector, partial_trace, state_fidelity
from qiskit_aer import AerSimulator


def show_plot():
    import io

    from IPython.display import Image, display

    buffer = io.BytesIO()
    plt.gcf().savefig(buffer, format="png", dpi=120, bbox_inches="tight")
    display(Image(data=buffer.getvalue()))
    plt.close()


phases = np.linspace(0, 2 * np.pi, 41)
probabilities = []
for phase in phases:
    qc = QuantumCircuit(1)
    qc.h(0)
    qc.rz(phase, 0)
    qc.h(0)
    probabilities.append(Statevector.from_instruction(qc).probabilities()[0])
assert np.allclose(probabilities, (1 + np.cos(phases)) / 2, atol=1e-12)
mixed = DensityMatrix(np.eye(2) / 2)
assert np.allclose(
    mixed.evolve(Operator(np.array([[1, 1], [1, -1]]) / np.sqrt(2))).probabilities(), [0.5, 0.5]
)
plt.plot(phases, probabilities, label="Coherent")
plt.axhline(0.5, color="gray", linestyle="--", label="Dephased")
plt.xlabel("Phase (radians)")
plt.ylabel("P(0)")
plt.legend()
show_plot()

## 2. Bell correlations violate the CHSH bound

Let $A_0=Z$, $A_1=X$, $B_0=(Z+X)/\sqrt2$, and $B_1=(Z-X)/\sqrt2$. On $|\Phi^+\rangle$, $S=E_{00}+E_{01}+E_{10}-E_{11}=2\sqrt2$, exceeding the local hidden-variable bound of 2. We compute the exact expectations and then sample synthetic ±1 outcomes from those probabilities. This is an ideal mathematical simulation, not a loophole-free experimental Bell test.


In [ ]:
X = np.array([[0, 1], [1, 0]], complex)
Z = np.diag([1, -1]).astype(complex)
bell = QuantumCircuit(2)
bell.h(0)
bell.cx(0, 1)
psi = Statevector.from_instruction(bell).data
A = [Z, X]
B = [(Z + X) / np.sqrt(2), (Z - X) / np.sqrt(2)]
expectations = np.array(
    [[np.vdot(psi, np.kron(B[b], A[a]) @ psi).real for b in range(2)] for a in range(2)]
)
signs = np.array([[1, 1], [1, -1]])
S = float(np.sum(signs * expectations))
assert abs(S - 2 * np.sqrt(2)) < 1e-12
rng = np.random.default_rng(23)
shots = 20000
sampled = 2 * rng.binomial(shots, (1 + expectations) / 2) / shots - 1
S_sampled = float(np.sum(signs * sampled))
# Hoeffding + union bound: each ±1 mean within .04 with failure <= 8 exp(-N*.04²/2).
assert np.max(np.abs(sampled - expectations)) < 0.04
assert S_sampled > 2
print({"CHSH_exact": S, "CHSH_sampled": S_sampled, "shots_per_setting": shots})

## 3. Teleport with two classical bits

Alice's message is $q_0$, her Bell half is $q_1$, and Bob holds $q_2$. After Alice applies CX$(0,1)$ and H$(0)$, measurements $m_0,m_1$ determine Bob's correction $Z^{m_0}X^{m_1}$. Applying X then Z is one valid ordering; the alternative differs only by a branch-dependent global phase. Bob cannot recover the message without receiving Alice's two classical bits, so no faster-than-light communication occurs.


In [ ]:
def preparation(label):
    qc = QuantumCircuit(1)
    if label == "1":
        qc.x(0)
    elif label == "+":
        qc.h(0)
    elif label == "-":
        qc.x(0)
        qc.h(0)
    elif label == "+i":
        qc.h(0)
        qc.s(0)
    elif label == "-i":
        qc.h(0)
        qc.sdg(0)
    elif label != "0":
        raise ValueError("Unknown state")
    return qc


def teleport(label, correct=True):
    qc = QuantumCircuit(3, 2)
    qc.compose(preparation(label), [0], inplace=True)
    qc.h(1)
    qc.cx(1, 2)
    qc.cx(0, 1)
    qc.h(0)
    qc.measure(0, 0)
    qc.measure(1, 1)
    if correct:
        with qc.if_test((qc.clbits[1], 1)):
            qc.x(2)
        with qc.if_test((qc.clbits[0], 1)):
            qc.z(2)
    qc.save_density_matrix()
    return qc


sim = AerSimulator(method="density_matrix")
labels = ["0", "1", "+", "-", "+i", "-i"]
circuits = [teleport(label) for label in labels] + [teleport("+", False)]
results = sim.run(
    transpile(circuits, sim, optimization_level=0, seed_transpiler=29),
    shots=8192,
    seed_simulator=29,
).result()
rows = []
for index, label in enumerate(labels):
    bob = partial_trace(DensityMatrix(results.data(index)["density_matrix"]), [0, 1])
    expected = Statevector.from_instruction(preparation(label))
    fidelity = state_fidelity(bob, expected)
    assert fidelity > 1 - 1e-10
    rows.append({"input": label, "output_fidelity": fidelity})
uncorrected = partial_trace(DensityMatrix(results.data(6)["density_matrix"]), [0, 1])
# Ignoring Alice's outcomes yields I/2 up to finite-shot branch averaging.
assert np.max(np.abs(uncorrected.data - np.eye(2) / 2)) < 0.04
print(pd.DataFrame(rows))
print("Uncorrected + input fidelity:", state_fidelity(uncorrected, Statevector.from_label("+")))

## 4. What do these tests establish?

The six inputs span the single-qubit operator space, and ideal corrected density matrices match their targets. The negative control verifies that classical corrections matter. The simulator can inspect Bob's density matrix directly; a real device needs measurements in several bases and uncertainty analysis. A Z-basis histogram alone cannot distinguish $|+\rangle$ from $|-\rangle$.


## Limits, cleanup, and next steps

This tutorial uses perfect gates, measurement, and feedforward. Finite-shot sampling checks only the stochastic experiment; it is not a noise estimate. For a hardware extension, first review current dynamic-circuit support, backend layout, transpilation, cost, and state-tomography methodology. No cloud credentials, course graders, or hardware jobs are used.

Restart the kernel to release figures and simulator results. As an extension, add a phase error to the entangled resource and compare the ±X, ±Y, and ±Z input fidelities.

Primary references: [Bennett et al., Teleporting an Unknown Quantum State](https://doi.org/10.1103/PhysRevLett.70.1895), [Clauser et al., Proposed Experiment to Test Local Hidden-Variable Theories](https://doi.org/10.1103/PhysRevLett.23.880), [IBM dynamic circuits](https://quantum.cloud.ibm.com/docs/en/guides/classical-feedforward-and-control-flow).

**Attribution:** This independent rewrite develops topics from Qiskit Global Summer School 2025. See [course attribution and Apache 2.0 notice](../ATTRIBUTION.md). No course artwork, grading service, saved account, or hardware results are included. This is not an official IBM publication and does not imply IBM endorsement.
